In [1]:
from robustraster import dask_cluster_manager

dask_cluster = dask_cluster_manager.DaskClusterManager()
dask_cluster.create_cluster(mode="full", threads_per_worker=1, n_workers=19, memory_limit="3GB")

Dask dashboard is available at: http://127.0.0.1:8787/status


In [2]:
# 3. Obtain the header information for the Earth Engine query and store it in an xarray object.
#    This code does not do a full query for the data (yet)! 

#    In this example, we are just querying some data from Landsat 8 imagery 
#    over a small watershed for demo purposes.

from robustraster import dataset_manager
import ee
import json
import xarray as xr
from dask.distributed import performance_report
import dask

json_key = r"R:\Users\adrianom.UNR\Downloads\robust-raster-cecdcc4b5fba.json"

# Although we authenticated Google Earth Engine on our Dask workers, we also need to authenticate
# Google Earth Engine on our local machine!
with open(json_key, 'r') as file:
    data = json.load(file)
credentials = ee.ServiceAccountCredentials(data["client_email"], json_key)
ee.Initialize(credentials = credentials, opt_url='https://earthengine-highvolume.googleapis.com')


WSDemo = ee.FeatureCollection("projects/robust-raster/assets/boundaries/WSDemoSHP_Albers")


#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2014-01-01', '2014-12-31')
ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterDate('2020-05-01', '2020-08-31')

xarray_data = xr.open_dataset(ic,
            engine='ee', 
            crs="EPSG:3310", 
            scale=30,
            geometry=WSDemo.geometry())

xarray_data = xarray_data.chunk({"time": 748, "X": 64, "Y": 128}).persist()

# For this demo, we do a basic NDVI calculation.
def compute_ndvi(df):
    # Perform your calculations
    df['ndvi'] = (df['SR_B5'] - df['SR_B4']) / (df['SR_B5'] + df['SR_B4'])
    return df


def user_function_wrapper(ds, user_func, *args, **kwargs):
    df_input = ds.to_dataframe().reset_index()
    df_output = user_func(df_input, *args, **kwargs)
    df_output = df_output.set_index(list(ds.dims))
    ds_output = df_output.to_xarray()
    return ds_output


test = xr.map_blocks(user_function_wrapper, 
                    xarray_data, 
                    args=(compute_ndvi,))

# Create a Dask report of the single chunked run
with performance_report(filename="dask_report.html"):
    test.compute()

TypeError: Function must return an xarray DataArray or Dataset. Instead it returned <class 'dask.delayed.Delayed'>